In [ ]:
#Installs and imports

In [ ]:
!pip install gymnasium[box2d] torch numpy matplotlib -q

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque
import matplotlib.pyplot as plt

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
        )

    def forward(self, x):
        return self.net(x)

replay buffer

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity=50000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = zip(*batch)
        return (np.array(state), action, reward, np.array(next_state), done)

    def __len__(self):
        return len(self.buffer)

setup

In [ ]:
env = gym.make("LunarLander-v3")  # use "LunarLander-v2" if v3 isn't available in your gym version
state_dim = env.observation_space.shape[0]   # 8 for Lunar Lander
action_dim = env.action_space.n              # 4 discrete actions

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

policy_net = QNetwork(state_dim, action_dim).to(device)
target_net = QNetwork(state_dim, action_dim).to(device)
target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.Adam(policy_net.parameters(), lr=5e-4)
buffer = ReplayBuffer()

gamma = 0.99
batch_size = 64
epsilon = 1.0
epsilon_min = 0.01
epsilon_decay = 0.995
tau = 0.005  # soft update rate

action selection

In [ ]:
def select_action(state, epsilon):
    if random.random() < epsilon:
        return env.action_space.sample()
    with torch.no_grad():
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        q_values = policy_net(state_t)
        return q_values.argmax().item()

training step

In [ ]:
def train_step():
    if len(buffer) < batch_size:
        return

    states, actions, rewards, next_states, dones = buffer.sample(batch_size)

    states = torch.FloatTensor(states).to(device)
    actions = torch.LongTensor(actions).unsqueeze(1).to(device)
    rewards = torch.FloatTensor(rewards).unsqueeze(1).to(device)
    next_states = torch.FloatTensor(next_states).to(device)
    dones = torch.FloatTensor(dones).unsqueeze(1).to(device)

    q_values = policy_net(states).gather(1, actions)
    with torch.no_grad():
        max_next_q = target_net(next_states).max(1, keepdim=True)[0]
        target_q = rewards + gamma * max_next_q * (1 - dones)

    loss = nn.SmoothL1Loss()(q_values, target_q)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 1.0)
    optimizer.step()

    # Soft update target network
    for target_param, policy_param in zip(target_net.parameters(), policy_net.parameters()):
        target_param.data.copy_(tau * policy_param.data + (1 - tau) * target_param.data)

training loop

In [ ]:
num_episodes = 700
reward_history = []

for episode in range(num_episodes):
    state, _ = env.reset()
    total_reward = 0
    done = False

    while not done:
        action = select_action(state, epsilon)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        buffer.push(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward

        train_step()

    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    reward_history.append(total_reward)

    if episode % 20 == 0:
        avg_reward = np.mean(reward_history[-20:])
        print(f"Episode {episode}, Reward: {total_reward:.1f}, Avg(last 20): {avg_reward:.1f}, Epsilon: {epsilon:.3f}")

Episode 0, Reward: -139.7, Avg(last 20): -139.7, Epsilon: 0.995
Episode 20, Reward: -286.3, Avg(last 20): -173.2, Epsilon: 0.900
Episode 40, Reward: -80.2, Avg(last 20): -113.0, Epsilon: 0.814
Episode 60, Reward: -99.5, Avg(last 20): -76.7, Epsilon: 0.737
Episode 80, Reward: 8.0, Avg(last 20): -78.1, Epsilon: 0.666
Episode 100, Reward: -59.2, Avg(last 20): -68.6, Epsilon: 0.603
Episode 120, Reward: -188.1, Avg(last 20): -65.2, Epsilon: 0.545
Episode 140, Reward: 13.5, Avg(last 20): -28.7, Epsilon: 0.493
Episode 160, Reward: -69.1, Avg(last 20): -29.2, Epsilon: 0.446
Episode 180, Reward: -15.0, Avg(last 20): -29.6, Epsilon: 0.404
Episode 200, Reward: 102.0, Avg(last 20): 34.1, Epsilon: 0.365
Episode 220, Reward: 12.3, Avg(last 20): 52.6, Epsilon: 0.330
Episode 240, Reward: -47.7, Avg(last 20): 0.1, Epsilon: 0.299
Episode 260, Reward: -210.0, Avg(last 20): 58.0, Epsilon: 0.270
Episode 280, Reward: 61.3, Avg(last 20): 48.2, Epsilon: 0.245
Episode 300, Reward: 167.4, Avg(last 20): 88.1, Ep

In [ ]:
plt.plot(reward_history)
plt.axhline(y=200, color='r', linestyle='--', label='Solved threshold (200)')
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("Lunar Lander DQN Training Progress")
plt.legend()
plt.show()

random baseline

In [ ]:
def random_agent_baseline(episodes=5):
    rewards = []
    for _ in range(episodes):
        state, _ = env.reset()
        done = False
        total = 0
        while not done:
            action = env.action_space.sample()
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total += reward
        rewards.append(total)
    print(f"Random agent avg reward: {np.mean(rewards):.1f}")
    return rewards

random_rewards = random_agent_baseline()

Random agent avg reward: -151.8


test trained agents

In [ ]:
def test_agent(episodes=5):
    for ep in range(episodes):
        state, _ = env.reset()
        done = False
        total_reward = 0
        while not done:
            with torch.no_grad():
                state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
                action = policy_net(state_t).argmax().item()
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward
        print(f"Test Episode {ep+1}: Reward = {total_reward:.1f}")

test_agent()

Test Episode 1: Reward = 286.8
Test Episode 2: Reward = 243.0
Test Episode 3: Reward = 281.1
Test Episode 4: Reward = 258.4
Test Episode 5: Reward = 291.0
